In [42]:
import sys
sys.path.append("../")
from chess_engine.src.model.classes.sqlite.models import (GamePositionRollup)
from chess_engine.src.model.classes.sqlite.database import SessionLocal
from chess_engine.src.model.classes.bitboard_processing.bitboard_creator import get_all_bitboards_dict
from chess_engine.src.model.config.config import Settings
from typing import List, Tuple
from sqlalchemy import or_, and_, func
import chess
import re
import pandas as pd
import numpy as np
import json 
from tqdm import tqdm
from sqlalchemy import or_, and_, func
from chess_engine.src.model.classes.bitboard_processing.bitboard_creator import bitboards_to_array, sample_bitboard_dict, Bitboard_Creator
from chess_engine.src.model.classes.sqlite.database import  get_db
from chess_engine.src.model.classes.sqlite.models import GamePositionRollup
from chess_engine.src.model.classes.bitboard_processing.bitboard_creator import bitboards_to_array, sample_bitboard_dict, Bitboard_Creator
from chess_engine.src.model.classes.sqlite.database import  get_db
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch
import os
# src\model\classes\npz_piping\create_npz_files.py
from chess_engine.src.model.classes.npz_piping.create_npz_files import db_to_npz_files
from chess_engine.src.model.config.config import settings

In [43]:
db_to_npz_files()

Deleted: ./src/model/data/training\data_0.npz
Deleted: ./src/model/data/training\data_1.npz
Deleted: ./src/model/data/training\data_10.npz
Deleted: ./src/model/data/training\data_11.npz
Deleted: ./src/model/data/training\data_12.npz
Deleted: ./src/model/data/training\data_13.npz
Deleted: ./src/model/data/training\data_14.npz
Deleted: ./src/model/data/training\data_15.npz
Deleted: ./src/model/data/training\data_16.npz
Deleted: ./src/model/data/training\data_17.npz
Deleted: ./src/model/data/training\data_18.npz
Deleted: ./src/model/data/training\data_19.npz
Deleted: ./src/model/data/training\data_2.npz
Deleted: ./src/model/data/training\data_20.npz
Deleted: ./src/model/data/training\data_21.npz
Deleted: ./src/model/data/training\data_22.npz
Deleted: ./src/model/data/training\data_23.npz
Deleted: ./src/model/data/training\data_24.npz
Deleted: ./src/model/data/training\data_25.npz
Deleted: ./src/model/data/training\data_26.npz
Deleted: ./src/model/data/training\data_27.npz
Failed to delete

In [70]:
import os
import bisect
import torch
from torch.utils.data import Dataset
import numpy as np

class NpzDataset(Dataset):
    def __init__(self, data_directory, transform=None, target_transform=None):
        """
        Custom Dataset for loading data from multiple .npz files.

        Args:
            data_directory (str): Directory containing the .npz files.
            transform (callable, optional): Optional transform to be applied
                on a sample.
            target_transform (callable, optional): Optional transform to be applied
                on the target.
        """
        self.data_directory = data_directory
        self.transform = transform
        self.target_transform = target_transform
        
        self.files = []
        self.file_sample_counts = []
        self.cumulative_counts = [0]  # Start with 0 to correctly index the first file
        self.file_cache = {}
        self.max_cache_size = 5  # Adjust based on available memory
        
        total_samples = 0
        for file_name in sorted(os.listdir(data_directory)):
            if file_name.endswith('.npz'):
                file_path = os.path.join(data_directory, file_name)
                self.files.append(file_path)
                
                # Load only the header to get the number of samples
                with np.load(file_path) as data:
                    n_samples = data['features'].shape[0]
                self.file_sample_counts.append(n_samples)
                total_samples += n_samples
                self.cumulative_counts.append(total_samples)
        
        self.total_samples = total_samples

    def __len__(self):
        return self.total_samples

    def __getitem__(self, idx):
        if idx < 0 or idx >= self.total_samples:
            raise IndexError(f"Index {idx} out of bounds for dataset of size {self.total_samples}")
        
        # Find the file index using binary search
        file_idx = bisect.bisect_right(self.cumulative_counts, idx) - 1
        sample_idx = idx - self.cumulative_counts[file_idx]

        file_path = self.files[file_idx]

        # Use caching to avoid reloading the same file
        if file_path in self.file_cache:
            data = self.file_cache[file_path]
        else:
            if len(self.file_cache) >= self.max_cache_size:
                # Remove the first cached file (simple FIFO cache)
                removed_file = next(iter(self.file_cache))
                del self.file_cache[removed_file]
            data = np.load(file_path)
            self.file_cache[file_path] = data

        features = data['features']
        labels = data['labels']

        # Check if sample_idx is within the bounds of the data arrays
        if sample_idx < 0 or sample_idx >= features.shape[0]:
            raise IndexError(f"Sample index {sample_idx} out of bounds for file {file_path} with size {features.shape[0]}")

        feature_sample = features[sample_idx]
        label_sample = labels[sample_idx]

        # Convert features to tensor
        if self.transform:
            feature_sample = self.transform(feature_sample)
        else:
            feature_sample = torch.from_numpy(feature_sample).float()
        
        # Convert labels from one-hot encoding to class indices
        if self.target_transform:
            label_sample = self.target_transform(label_sample)
        else:
            # label_sample is one-hot encoded, convert to class index
            label_sample = torch.from_numpy(label_sample).long()
            label_sample = torch.argmax(label_sample)

        return feature_sample, label_sample


In [59]:
# Test the dataset indexing
errors_found = False
for idx in range(len(dataset)):
    try:
        features, labels = dataset[idx]
    except Exception as e:
        print(f"Error at index {idx}: {e}")
        errors_found = True
        break

if not errors_found:
    print("All indices accessed successfully.")


All indices accessed successfully.


In [68]:


# Replace 'your_training_directory' with the actual path to your training data directory
dataset = NpzDataset(settings.npzTrainingDirectory)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=0)

# Test the dataset indexing
for idx in range(len(dataset)):
    try:
        features, labels = dataset[idx]
    except Exception as e:
        print(f"Error at index {idx}: {e}")
        break

# Iterate through the dataloader
for batch_features, batch_labels in dataloader:

    pass


In [71]:
import os
import bisect
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

# Define the NpzDataset class
class NpzDataset(Dataset):
    def __init__(self, data_directory, transform=None, target_transform=None):
        """
        Custom Dataset for loading data from multiple .npz files.

        Args:
            data_directory (str): Directory containing the .npz files.
            transform (callable, optional): Optional transform to be applied
                on a sample.
            target_transform (callable, optional): Optional transform to be applied
                on the target.
        """
        self.data_directory = data_directory
        self.transform = transform
        self.target_transform = target_transform
        
        self.files = []
        self.file_sample_counts = []
        self.cumulative_counts = [0]  # Start with 0 to correctly index the first file
        self.file_cache = {}
        self.max_cache_size = 5  # Adjust based on available memory
        
        total_samples = 0
        for file_name in sorted(os.listdir(data_directory)):
            if file_name.endswith('.npz'):
                file_path = os.path.join(data_directory, file_name)
                self.files.append(file_path)
                
                # Load only the header to get the number of samples
                with np.load(file_path) as data:
                    n_samples = data['features'].shape[0]
                self.file_sample_counts.append(n_samples)
                total_samples += n_samples
                self.cumulative_counts.append(total_samples)
        
        self.total_samples = total_samples

    def __len__(self):
        return self.total_samples

    def __getitem__(self, idx):
        if idx < 0 or idx >= self.total_samples:
            raise IndexError(f"Index {idx} out of bounds for dataset of size {self.total_samples}")
        
        # Find the file index using binary search
        file_idx = bisect.bisect_right(self.cumulative_counts, idx) - 1
        sample_idx = idx - self.cumulative_counts[file_idx]

        file_path = self.files[file_idx]

        # Use caching to avoid reloading the same file
        if file_path in self.file_cache:
            data = self.file_cache[file_path]
        else:
            if len(self.file_cache) >= self.max_cache_size:
                # Remove the first cached file (simple FIFO cache)
                removed_file = next(iter(self.file_cache))
                del self.file_cache[removed_file]
            data = np.load(file_path)
            self.file_cache[file_path] = data

        features = data['features']
        labels = data['labels']

        # Check if sample_idx is within the bounds of the data arrays
        if sample_idx < 0 or sample_idx >= features.shape[0]:
            raise IndexError(f"Sample index {sample_idx} out of bounds for file {file_path} with size {features.shape[0]}")

        feature_sample = features[sample_idx]
        label_sample = labels[sample_idx]

        # Convert features to tensor
        if self.transform:
            feature_sample = self.transform(feature_sample)
        else:
            feature_sample = torch.from_numpy(feature_sample).float()
        
        # Convert labels from one-hot encoding to class indices
        if self.target_transform:
            label_sample = self.target_transform(label_sample)
        else:
            # label_sample is one-hot encoded, convert to class index
            label_sample = torch.from_numpy(label_sample).long()
            label_sample = torch.argmax(label_sample)

        return feature_sample, label_sample

# Define the SimpleCNN model
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(12, 32, kernel_size=3, padding=1),  # Output: (32, 8, 8)
            nn.ReLU(),
            nn.MaxPool2d(2),  # Output: (32, 4, 4)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # Output: (64, 4, 4)
            nn.ReLU(),
            nn.MaxPool2d(2)   # Output: (64, 2, 2)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 2 * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 3)  # Output size matches number of classes
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

# Set up datasets and data loaders
# Ensure that settings.npzTrainingDirectory, settings.npzValidationDirectory, and settings.npzTestingDirectory are set
train_dataset = NpzDataset(settings.npzTrainingDirectory)
val_dataset = NpzDataset(settings.npzValidationDirectory)
test_dataset = NpzDataset(settings.npzTestingDirectory)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

# Initialize model, loss function, optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for batch_features, batch_labels in train_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_features)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * batch_features.size(0)
    
    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {epoch_loss:.4f}")
    
    # Validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for val_features, val_labels in val_loader:
            val_features = val_features.to(device)
            val_labels = val_labels.to(device)
            
            outputs = model(val_features)
            loss = criterion(outputs, val_labels)
            val_loss += loss.item() * val_features.size(0)
            
            _, predicted = torch.max(outputs.data, 1)
            total += val_labels.size(0)
            correct += (predicted == val_labels).sum().item()
    
    val_epoch_loss = val_loss / len(val_dataset)
    val_accuracy = correct / total
    print(f"Validation Loss: {val_epoch_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

# Testing the model
model.eval()
test_loss = 0.0
correct = 0
total = 0
with torch.no_grad():
    for test_features, test_labels in test_loader:
        test_features = test_features.to(device)
        test_labels = test_labels.to(device)
        
        outputs = model(test_features)
        loss = criterion(outputs, test_labels)
        test_loss += loss.item() * test_features.size(0)
        
        _, predicted = torch.max(outputs.data, 1)
        total += test_labels.size(0)
        correct += (predicted == test_labels).sum().item()

test_epoch_loss = test_loss / len(test_dataset)
test_accuracy = correct / total
print(f"Test Loss: {test_epoch_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")


Epoch 1/10, Training Loss: 0.9567
Validation Loss: 0.8079, Validation Accuracy: 0.6286
Epoch 2/10, Training Loss: 0.5638
Validation Loss: 0.4564, Validation Accuracy: 0.8107
Epoch 3/10, Training Loss: 0.3282
Validation Loss: 0.3151, Validation Accuracy: 0.8750
Epoch 4/10, Training Loss: 0.2026
Validation Loss: 0.2153, Validation Accuracy: 0.9286
Epoch 5/10, Training Loss: 0.1468
Validation Loss: 0.2096, Validation Accuracy: 0.9357
Epoch 6/10, Training Loss: 0.1049
Validation Loss: 0.1911, Validation Accuracy: 0.9179
Epoch 7/10, Training Loss: 0.0854
Validation Loss: 0.2260, Validation Accuracy: 0.9286
Epoch 8/10, Training Loss: 0.0809
Validation Loss: 0.1549, Validation Accuracy: 0.9536
Epoch 9/10, Training Loss: 0.0594
Validation Loss: 0.1640, Validation Accuracy: 0.9429
Epoch 10/10, Training Loss: 0.0499
Validation Loss: 0.1718, Validation Accuracy: 0.9429
Test Loss: 0.0844, Test Accuracy: 0.9755


In [72]:
# Test the dataset and data loader
for batch_features, batch_labels in train_loader:
    print(f"Batch features shape: {batch_features.shape}")  # Should be (batch_size, 12, 8, 8)
    print(f"Batch labels shape: {batch_labels.shape}")      # Should be (batch_size,)
    break  # Remove this after verification

# Test a forward pass
batch_features = batch_features.to(device)
outputs = model(batch_features)
print(f"Model output shape: {outputs.shape}")  # Should be (batch_size, 3)


Batch features shape: torch.Size([64, 12, 8, 8])
Batch labels shape: torch.Size([64])
Model output shape: torch.Size([64, 3])
